# Robo3 wS1 manuscript figures — NWB pipeline (Python)

This notebook regenerates every figure the NWB pipeline reproduces, calling each
plotting function in turn. All outputs are written under `NWBData\Figures\...`
and every file is named by its **manuscript figure and panel** plus what it shows,
e.g. `Fig3D - Mean spike rate contra vs ipsi (window 150 ms).pdf`.

| Script | Panels produced |
|---|---|
| `fig1_behav_nwb` | **Fig1D** correct/miss · **FigS1A** amplitude · **FigS1B** correct L/R · **FigS1C** miss L/R |
| `fig1_learning_nwb` | **FigS1D** learning rate |
| `fig2_optoinhib_nwb` | **Fig2B** Δcorrect · **Fig2C** Δmiss · **FigS2B/C** raw performance · **FigS2D** amplitude |
| `fig3_tables_nwb` | accumulation tables (`fig3_tables_S1.mat` / `.csv`) |
| `fig3_panels_nwb` | **Fig3C** heatmap · **Fig3D** mean spike rate · **Fig3E** laterality · **Fig3F** percent contra |
| `figS3S4_nwb` & `figS4a_histology_nwb` | **FigS3A/B** per-freq heatmaps · **FigS4A** 3D CCF histology · **FigS4B** layer preference + bilateral fraction |
| `fig4_roc_nwb` | **Fig4A** 50ms AUC hist · **Fig4B** 5ms selectivity · **Fig4C** onset CDF + stats · **FigS5** perm/boot |
| `fig5_lda_nwb` | **Fig5A** session-wise accuracy · **Fig5B** bootstrapped accuracy · **Fig5** composite |
| `fig6_panels_nwb` & `fig6_roc_nwb` | **Fig6A/B** heatmaps & mean rates · **Fig6C** 50ms AUC hist · **Fig6D_E** 5ms selectivity · **Fig6F** onset CDF · **FigS8** perm/boot |
| `fig7_lda_nwb` | **Fig7A** session-wise accuracy · **Fig7B** bootstrapped accuracy · **Fig7** composite |
| `figS6_nwb` & `figS6b_histology_nwb` | **FigS6B** dorsal view probe entry · **FigS6C** depth preference · **FigS6D** S1 vs M1 onset comparison |

Histology panels (**Fig S4A** 3D CCF coordinates and **Fig S6B** dorsal probe entry points) are reproduced from standardized coordinate staging tables.


## 1 — Configuration
Edit `BASE` (and, for the learning panel, `SESSINFO_XLSX`) to match your machine, then run every cell top to bottom.

In [13]:
import os, sys
from pathlib import Path

# --- root of the NWB release (edit for your machine) ---
BASE = Path(r"E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData")

# The ported functions live in THIS notebook's own folder (pyWSD).
# Jupyter's working directory is the notebook's folder, so add it to the path.
CODE_DIR = Path.cwd()
sys.path.insert(0, str(CODE_DIR))

# --- input data folders ---
DATA          = BASE / "Data"
EXPERT_DIR    = DATA / "behavior" / "expert"
LEARNING_DIR  = DATA / "behavior" / "learning"
OPTO_DIR      = DATA / "behavior" / "opto"
S1_EPHYS      = DATA / "ephys" / "S1"
M1_EPHYS      = DATA / "ephys" / "M1"
TABLES_DIR    = DATA / "Tables"

# The learning panel needs the per-animal training-window spreadsheet (the ONLY
# non-NWB input). Point this at your LearningSessInfo\SessionInfo.xlsx.
SESSINFO_XLSX = LEARNING_DIR / "SessionInfo.xlsx"

# --- output figure folders (Python pipeline -> NWBData\Figures\Python\...) ---
FIGURES    = BASE / "Figures" / "Python"
FIG1_DIR   = FIGURES / "Fig1_FigS1"
FIG2_DIR   = FIGURES / "Fig2"
FIGS2_DIR  = FIGURES / "FigS2"
FIG3_DIR   = FIGURES / "Fig3"
FIGS34_DIR = FIGURES / "FigS3_FigS4"
FIG4_DIR   = FIGURES / "Fig4"
FIG5_DIR   = FIGURES / "Fig5"
FIG6_DIR   = FIGURES / "Fig6"
FIGS6_DIR  = FIGURES / "FigS6"
FIG7_DIR   = FIGURES / "Fig7"
LDA_TABLES_DIR = TABLES_DIR / "LDA"
ROC_TABLES_DIR = TABLES_DIR / "ROC"
HISTOLOGY_TABLES_DIR = TABLES_DIR / "Histology"
for d in (FIG1_DIR, FIG2_DIR, FIGS2_DIR, FIG3_DIR, FIGS34_DIR, FIG4_DIR, FIG5_DIR, FIG6_DIR, FIG7_DIR, FIGS6_DIR, TABLES_DIR, ROC_TABLES_DIR, LDA_TABLES_DIR, HISTOLOGY_TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

TABLES_MAT    = TABLES_DIR / "fig3_tables_S1.mat"
TABLES_MAT_M1 = TABLES_DIR / "fig3_tables_M1.mat"
print("BASE:", BASE)
print("Code dir on path:", CODE_DIR)


BASE: E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData
FinalCodes on path: True


## 2 — Figure 1 / S1 — behavioral performance
`fig1_behav_nwb` writes **Fig1D** (fraction correct / miss) and **FigS1A/B/C** (amplitude, and left/right splits).

In [9]:
import fig1_behav_nwb
fig1_behav_nwb.main(["--nwb-dir", str(EXPERT_DIR), "--out-dir", str(FIG1_DIR)])


loaded 62 bodyside6 expert sessions across 18 mice (excluded pilot cohort ['VC030103', 'VC030104', 'VC030201', 'VC030202'])
WT mice (10): ['VC030105', 'VC030106', 'VC030107', 'VC030108', 'VC030109', 'VC030110', 'VC030112', 'VC030113', 'VC030114', 'VC030115']
KO mice (8): ['VC030203', 'VC030204', 'VC030206', 'VC030207', 'VC030208', 'VC030209', 'VC030211', 'VC030213']
wrote 8 panels + Fig1D - behavior mean values.txt to E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Figures\Python\Fig1_FigS1


0

### Figure S1D — learning rate
Needs `SESSINFO_XLSX` (the training-window spreadsheet). Skip this cell if you do not have it.

In [3]:
import fig1_learning_nwb
if SESSINFO_XLSX.exists():
    fig1_learning_nwb.main(["--nwb-dir", str(LEARNING_DIR),
                            "--sessinfo", str(SESSINFO_XLSX),
                            "--out-dir", str(FIG1_DIR)])
else:
    print("SessionInfo.xlsx not found at", SESSINFO_XLSX, "- skipping FigS1D (learning).")


SessionInfo.xlsx not found at E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Data\behavior\learning\SessionInfo.xlsx - skipping FigS1D (learning).


## 3 — Figure 2 / S2 — optogenetic inhibition
`fig2_optoinhib_nwb` writes **Fig2B** (Δcorrect), **Fig2C** (Δmiss) and **FigS2B/C/D** (raw performance, amplitude).

In [4]:
import fig2_optoinhib_nwb
fig2_optoinhib_nwb.main(["--nwb-dir", str(OPTO_DIR),
                         "--fig2-dir", str(FIG2_DIR),
                         "--figs2-dir", str(FIGS2_DIR)])


loaded 102 opto sessions
  WT Right S1: 5 mice ['VC030109', 'VC030110', 'VC030112', 'VC030114', 'VC030115']
  WT Left S1: 5 mice ['VC030109', 'VC030110', 'VC030112', 'VC030114', 'VC030115']
  KO Right S1: 5 mice ['VC030207', 'VC030208', 'VC030209', 'VC030211', 'VC030213']
  KO Left S1: 5 mice ['VC030207', 'VC030208', 'VC030209', 'VC030211', 'VC030213']
Bootstrapping per mouse (10000 iterations each) ...
  WT / Right S1 / VC030109 (5 sessions) ...
  WT / Right S1 / VC030110 (5 sessions) ...
  WT / Right S1 / VC030112 (5 sessions) ...
  WT / Right S1 / VC030114 (5 sessions) ...
  WT / Right S1 / VC030115 (5 sessions) ...
  WT / Left S1 / VC030109 (5 sessions) ...
  WT / Left S1 / VC030110 (5 sessions) ...
  WT / Left S1 / VC030112 (5 sessions) ...
  WT / Left S1 / VC030114 (5 sessions) ...
  WT / Left S1 / VC030115 (5 sessions) ...
  KO / Right S1 / VC030207 (5 sessions) ...
  KO / Right S1 / VC030208 (5 sessions) ...
  KO / Right S1 / VC030209 (6 sessions) ...
  KO / Right S1 / VC030211

0

## 4 — Accumulate the Fig3 tables (once)
`fig3_tables_nwb` reads the S1 ephys NWBs once and writes `fig3_tables_S1.mat` / `.csv`, which every ephys panel then reads.

In [5]:
import fig3_tables_nwb
fig3_tables_nwb.main(["--region-dir", str(S1_EPHYS), "--region", "S1", "--out-dir", str(TABLES_DIR)])
print("tables ->", TABLES_MAT)


accumulating S1 tables from E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Data\ephys\S1
  VC030107_2022-11-15a.nwb: 4 unit-freq records (WT Left wS1)
  VC030107_2022-11-16a.nwb: 15 unit-freq records (WT Left wS1)
  VC030109_2023-03-02a.nwb: 40 unit-freq records (WT Left wS1)
  VC030109_2023-03-02b.nwb: 28 unit-freq records (WT Left wS1)
  VC030109_2023-03-04a.nwb: 12 unit-freq records (WT Left wS1)
  VC030109_2023-03-07a.nwb: 136 unit-freq records (WT Right wS1)
  VC030109_2023-03-08a.nwb: 12 unit-freq records (WT Right wS1)
  VC030109_2023-03-09a.nwb: 48 unit-freq records (WT Right wS1)
  VC030112_2023-05-22a.nwb: 104 unit-freq records (WT Left wS1)
  VC030112_2023-05-23a.nwb: 124 unit-freq records (WT Left wS1)
  VC030112_2023-05-24a.nwb: 40 unit-freq records (WT Left wS1)
  VC030112_2023-05-26a.nwb: 80 unit-freq records (WT Right wS1)
  VC030112_2023-05-27a.nwb: 40 unit-freq records (WT Right wS1)
  VC030113_2023-08-21a.nwb: 68 unit-freq records (WT Le

## 5 — Build the unit table (shared by every ephys panel)

In [6]:
import fig3_panels_nwb as fp
import figS3S4_nwb as fs

records   = fp.loadFlatTable(str(TABLES_MAT))
unitTable = fp.buildUnitTable(records)
nResp = sum(1 for r in unitTable if r["responsive"])
print(f"{len(unitTable)} units ({nResp} responsive)")


1208 units (177 responsive)


## 6 — Figure 3 panels
Each call writes one panel set, named by its manuscript figure.

In [7]:
fp.frHeatmapsAllFreq(unitTable, str(FIG3_DIR))      # Fig3C - Normalized FR heatmap WT/KO
fp.plotAvgTracesFRFinal(unitTable, str(FIG3_DIR))   # Fig3D - Mean spike rate contra vs ipsi
fp.makeCbiasFiguresFinal(unitTable, str(FIG3_DIR))  # Fig3E - Laterality index / Fig3F - Percent contra
print("Fig3 ->", FIG3_DIR)


Fig3 -> E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Figures\Python\Fig3


## 7 — Figure S3 / S4 panels

In [8]:
import figS4a_histology_tables_nwb
import figS4a_histology_nwb

if not (HISTOLOGY_TABLES_DIR / "figS4a_histology_S1.json").exists():
    figS4a_histology_tables_nwb.main(["--out-dir", str(HISTOLOGY_TABLES_DIR)])

figS4a_histology_nwb.generate_figS4a(HISTOLOGY_TABLES_DIR, FIGS34_DIR)      # FigS4A - 3D CCF unit locations
fs.frHeatmapsEachFreq(unitTable, str(FIGS34_DIR))         # FigS3A/B - per-frequency heatmaps
fs.cBiasDepthHistoLine(unitTable, str(FIGS34_DIR))        # FigS4B - Layer preference distribution
fs.bilateralFractionL4vsOther(unitTable, str(FIGS34_DIR)) # FigS4B - Bilateral fraction L4 vs other
print("FigS3/S4 ->", FIGS34_DIR)


FigS3/S4 -> E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Figures\Python\FigS3_FigS4


## 8 — Figure 4 / S5 — S1 Single-Unit ROC / Body-Side Selectivity
ig4_roc_nwb reads the staged 50-ms and 5-ms ROC tables and plots:
- **Fig 4A**: 50-ms AUC distribution histograms (WT and KO)
- **Fig 4B_C**: 3-panel bootstrapped 5-ms selectivity time course (WT, KO, and WT - KO difference)
- **Fig 4D**: Onset of significant side selectivity CDF & KS tests
- **Fig S5**: 50-ms bin bootstrapping confidence intervals & permutation tests (3x3 grid)


In [18]:
import importlib
import fig4_roc_nwb
importlib.reload(fig4_roc_nwb)
fig4_roc_nwb.main(["--tables-dir", str(ROC_TABLES_DIR),
                   "--out-dir", str(FIG4_DIR)])


Loading 50-ms ROC table: E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Data\Tables\ROC\fig4_roc_tables_S1_50ms.mat
Loading 5-ms ROC table: E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Data\Tables\ROC\fig4_roc_tables_S1_5ms.mat
Plotting Figure 4A (50-ms AUC distribution histograms)...
Plotting Figure 4B/C (5-ms selectivity time course, 3-panel bootstrapped)...
Plotting Figure 4D (Selectivity onset latency CDF and KS tests)...
Plotting Figure S5 (50-ms permutation and bootstrap tests, 3x3)...
Figure 4 and Figure S5 plots completed successfully in: E:\oconnorlab Dropbox\oconnorlab Team Folder\manuscripts\bodyside_S1\NWBData\Figures\Python\Fig4


## 9 — Figure 5 — Whisker-Side Stimulus Discriminability in wS1 (Population LDA Decoding)
Reads staged wS1 LDA tables (`fig5_lda_tables_S1.mat` / `.json`) to plot:
- **Fig 5A**: Session-wise classification accuracy distributions for WT ($n=14$) and KO ($n=16$) at -50~0 ms and 0~50 ms
- **Fig 5B**: Bootstrapped population stimulus-discriminability (mean ± 95% CI)
- **Fig 5 Composite**: Full 3x2 composite figure matching published `FinalFig5.png`
- **Statistical exports**: `LDA median values.txt` and `LDA bootstrapping CI values.txt`


In [ ]:
import importlib
import fig5_lda_tables_nwb
import fig5_lda_nwb
importlib.reload(fig5_lda_nwb)

if not (LDA_TABLES_DIR / "fig5_lda_tables_S1.mat").exists():
    fig5_lda_tables_nwb.main(["--out-dir", str(LDA_TABLES_DIR)])

fig5_lda_nwb.main([
    "--tables-dir", str(LDA_TABLES_DIR),
    "--out-dir", str(FIG5_DIR)
])


## 10 — Figure 6A & 6B — wM1 Normalized FR Heatmaps and Mean Spiking Rates
Reads `fig3_tables_M1.mat` and `responsive_units_M1.csv` to plot:
- **Fig 6A**: Normalized spiking rate heatmaps for responsive units (WT $n=128$, KO $n=148$)
- **Fig 6B**: Population mean spiking rate traces (10 Hz, 20 Hz, 40 Hz) with 95% across-trial CI error bands


In [ ]:
import importlib
import fig3_tables_nwb
import fig6_panels_nwb
importlib.reload(fig6_panels_nwb)

if not TABLES_MAT_M1.exists():
    fig3_tables_nwb.main(["--region-dir", str(M1_EPHYS), "--region", "M1", "--out-dir", str(TABLES_DIR)])

fig6_panels_nwb.main([
    "--tables", str(TABLES_MAT_M1),
    "--resp-csv", str(ROC_TABLES_DIR / "responsive_units_M1.csv"),
    "--out-dir", str(FIG6_DIR)
])


## 11 — Figure 6C-F & Figure S8 — wM1 Single-Unit ROC & Bootstrap/Permutation Tests
Reads staged M1 ROC tables (`fig6_roc_tables_M1_50ms.mat` and `fig6_roc_tables_M1_5ms.mat`) to plot:
- **Fig 6C**: 50-ms AUC distribution histograms for WT ($n=128$) and KO ($n=148$)
- **Fig 6D_E**: 3-panel bootstrapped 5-ms selectivity time courses (WT, KO, and $\Delta(\text{WT} - \text{KO})$)
- **Fig 6F**: Selectivity onset latency empirical CDF ($0\sim 80$ ms) & KS test statistics (`StatisticsSelectivityOnset.txt`)
- **Fig S8**: 50-ms bin bootstrapping confidence intervals (3x3 grid) matching `FinalFigS8.png` & `Statistics_permutationtest.txt`


In [ ]:
import importlib
import fig6_roc_tables_nwb
import fig6_roc_nwb
importlib.reload(fig6_roc_nwb)

if not (ROC_TABLES_DIR / "fig6_roc_tables_M1_50ms.mat").exists() or not (ROC_TABLES_DIR / "fig6_roc_tables_M1_5ms.mat").exists():
    fig6_roc_tables_nwb.main(["--out-dir", str(ROC_TABLES_DIR)])

fig6_roc_nwb.main([
    "--tables-dir", str(ROC_TABLES_DIR),
    "--out-dir", str(FIG6_DIR)
])


## 12 — Figure 7 — Whisker-Side Stimulus Discriminability in wM1 (Population LDA Decoding)
Reads staged wM1 LDA tables (`fig7_lda_tables_M1.mat` / `.json`) to plot:
- **Fig 7A**: Session-wise classification accuracy distributions for WT ($n=11$) and KO ($n=10$) at -100~0 ms and 0~100 ms
- **Fig 7B**: Bootstrapped population stimulus-discriminability (mean ± 95% CI)
- **Fig 7 Composite**: Full 3x2 composite figure matching published `FinalFig7.png`
- **Statistical exports**: `LDA median values.txt` and `LDA bootstrapping CI values.txt`


In [ ]:
import importlib
import fig7_lda_tables_nwb
import fig7_lda_nwb
importlib.reload(fig7_lda_nwb)

if not (LDA_TABLES_DIR / "fig7_lda_tables_M1.mat").exists():
    fig7_lda_tables_nwb.main(["--out-dir", str(LDA_TABLES_DIR)])

fig7_lda_nwb.main([
    "--tables-dir", str(LDA_TABLES_DIR),
    "--out-dir", str(FIG7_DIR)
])


## 13 — Figure S6 — wM1 Depth Laterality Preference & S1 vs M1 Selectivity Onset
Reads `fig3_tables_M1.mat` and staged ROC tables to plot:
- **Fig S6B**: Dorsal view of probe entry points in wM1 relative to Bregma (WT and KO)
- **Fig S6C**: Stimulus-side preference in wM1 by cortical depth (All units, Superficial 0-500 μm, Deep 500-1000 μm)
- **Fig S6D**: Empirical CDF comparing selectivity onset latencies between S1 and M1 for WT and KO mice
- **Statistics**: Exports `depth cbias histogram line stats.txt` and `Statistics.txt` (two-sample KS-tests)


In [ ]:
import importlib
import figS6b_histology_tables_nwb
import figS6_nwb
importlib.reload(figS6_nwb)

if not (HISTOLOGY_TABLES_DIR / "figS6b_histology_M1.json").exists():
    figS6b_histology_tables_nwb.main(["--out-dir", str(HISTOLOGY_TABLES_DIR)])

figS6_nwb.main([
    "--tables-m1", str(TABLES_MAT_M1),
    "--roc-dir", str(ROC_TABLES_DIR),
    "--histology-dir", str(HISTOLOGY_TABLES_DIR),
    "--resp-csv", str(ROC_TABLES_DIR / "responsive_units_M1.csv"),
    "--out-dir", str(FIGS6_DIR)
])


## Done
Every panel is in its `NWBData\Figures\...` folder, named by manuscript figure.
To regenerate a single figure, re-run its cell. To point at M1 instead of S1,
change `S1_EPHYS` / `--region` and rebuild the tables and unit table.